# Complete SFT → DPO Pipeline with Model Comparison



## 1. Installation & Setup

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install rouge-score bert-score

In [ ]:
print("✅ All packages installed successfully!")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192
dtype = None  # None for auto detection
load_in_4bit = True

# Load base model
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3-8B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=dtype
    # load_in_8bit=False,
    # dtype=torch.bfloat16,
)

# Set chat template
if base_tokenizer.chat_template is None:
    base_tokenizer.chat_template = "{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\\n\\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}{% endif %}"

print("✅ Base model loaded!")

## 2. Load Base Model

In [ ]:
max_seq_length = 2560

In [7]:
# from unsloth import FastLanguageModel
# import torch

# max_seq_length = 8192
# dtype = None  # None for auto detection
# load_in_4bit = True

# # Load base model
# base_model, base_tokenizer = FastLanguageModel.from_pretrained(
#     model_name="unsloth/Llama-3-8B",
#     max_seq_length=max_seq_length,
#     load_in_4bit=True,
#     # load_in_8bit=False,
#     # dtype=torch.bfloat16,
# )

# # Set chat template
# if base_tokenizer.chat_template is None:
#     base_tokenizer.chat_template = "{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\\n\\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}{% endif %}"

# print("✅ Base model loaded!")

ModuleNotFoundError: No module named 'unsloth'

## 3. Data Preparation & Splitting

In [3]:
import pandas as pd

df = pd.read_json("/kaggle/input/datasets/abhisheksaha1234/ami-and-qmsum/qmsum_spotlight_train.jsonl", lines=True)

# df.iloc[5000:38000].to_csv("subset_5000_33000.csv", index=False)

print("Done.")


Done.


In [ ]:
# Keep summaries ≤ 640 characters (≈160 tokens)
df = df[df['summary'].str.len() <= 640].reset_index(drop=True)
print(f"Dataset size after filtering: {len(df):,}")

In [ ]:
df.head()

In [4]:
df.shape

(596, 2)

In [ ]:
df["text"].iloc[0]

In [ ]:
# ============================================================================
# TEXT LENGTH ANALYSIS FOR FINE-TUNING
# Statistical analysis to determine optimal max_tokens and context window
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# ============================================================================
# LOAD YOUR DATA
# ============================================================================

# REPLACE THIS with your actual data loading code

print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

# ============================================================================
# CALCULATE LENGTHS
# ============================================================================

# Calculate character lengths
df['text_length'] = df['text'].str.len()
df['summary_length'] = df['summary'].str.len()

# Estimate token counts (rough estimate: 1 token ≈ 4 characters)
df['text_tokens'] = (df['text_length'] / 4).astype(int)
df['summary_tokens'] = (df['summary_length'] / 4).astype(int)
df['total_tokens'] = df['text_tokens'] + df['summary_tokens']

print("\n✓ Length calculations complete")
print(df[['text_tokens', 'summary_tokens', 'total_tokens']].head())

# ============================================================================
# QUICK SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*70)
print("QUICK FINE-TUNING PARAMETER GUIDE")
print("="*70)

print(f"\n📊 Dataset size: {len(df):,} examples")

print(f"\n📝 INPUT (text) lengths in tokens:")
print(f"   Mean:   {df['text_tokens'].mean():.0f}")
print(f"   Median: {df['text_tokens'].median():.0f}")
print(f"   95th:   {df['text_tokens'].quantile(0.95):.0f}")
print(f"   99th:   {df['text_tokens'].quantile(0.99):.0f}")
print(f"   Max:    {df['text_tokens'].max():.0f}")

print(f"\n✍️  OUTPUT (summary) lengths in tokens:")
print(f"   Mean:   {df['summary_tokens'].mean():.0f}")
print(f"   Median: {df['summary_tokens'].median():.0f}")
print(f"   95th:   {df['summary_tokens'].quantile(0.95):.0f}")
print(f"   99th:   {df['summary_tokens'].quantile(0.99):.0f}")
print(f"   Max:    {df['summary_tokens'].max():.0f}")

print(f"\n🎯 RECOMMENDED PARAMETERS:")

# Max tokens recommendation
recommended_max = int(df['summary_tokens'].quantile(0.95) * 1.3)
print(f"\n   max_tokens: {recommended_max}")
print(f"   (Covers 95% of outputs with 30% buffer)")

# Context window recommendation
print(f"\n   Context window needed: {int(df['text_tokens'].quantile(0.99))}")
print(f"   (To handle 99% of inputs)")

print("\n" + "="*70)

# ============================================================================
# DETAILED STATISTICS
# ============================================================================

print("\n\nTEXT (INPUT) TOKEN STATISTICS")
print("="*50)
print(df['text_tokens'].describe())
print(f"\n5th percentile:  {df['text_tokens'].quantile(0.05):.0f}")
print(f"95th percentile: {df['text_tokens'].quantile(0.95):.0f}")
print(f"99th percentile: {df['text_tokens'].quantile(0.99):.0f}")

print("\n\nSUMMARY (OUTPUT) TOKEN STATISTICS")
print("="*50)
print(df['summary_tokens'].describe())
print(f"\n5th percentile:  {df['summary_tokens'].quantile(0.05):.0f}")
print(f"95th percentile: {df['summary_tokens'].quantile(0.95):.0f}")
print(f"99th percentile: {df['summary_tokens'].quantile(0.99):.0f}")

# ============================================================================
# TRUNCATION ANALYSIS
# ============================================================================

print("\n\nTRUNCATION ANALYSIS")
print("="*50)
print("If you truncate INPUT texts to:")

truncation_options = [256, 512, 1024, 2048, 4096, 8192]
truncation_data = []

for length in truncation_options:
    pct_covered = (df['text_tokens'] <= length).mean() * 100
    print(f"   {length:5d} tokens → {pct_covered:5.1f}% of examples fit")
    truncation_data.append({'max_length': length, 'coverage': pct_covered})

truncation_df = pd.DataFrame(truncation_data)

# ============================================================================
# COMPRESSION RATIO ANALYSIS
# ============================================================================

compression_ratio = df['text_length'] / df['summary_length']

print("\n\nCOMPRESSION RATIO (text length / summary length)")
print("="*50)
print(f"Mean:   {compression_ratio.mean():.2f}x")
print(f"Median: {compression_ratio.median():.2f}x")
print(f"Min:    {compression_ratio.min():.2f}x")
print(f"Max:    {compression_ratio.max():.2f}x")
print(f"\nThis means summaries are typically {compression_ratio.mean():.1f}x shorter than the original text")

# ============================================================================
# VISUALIZATIONS
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Text Length Analysis for Fine-Tuning', fontsize=16, fontweight='bold')

# 1. Text length distribution
axes[0, 0].hist(df['text_tokens'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].axvline(df['text_tokens'].median(), color='red', linestyle='--', linewidth=2, 
                   label=f'Median: {df["text_tokens"].median():.0f}')
axes[0, 0].axvline(df['text_tokens'].quantile(0.95), color='orange', linestyle='--', linewidth=2, 
                   label=f'95th: {df["text_tokens"].quantile(0.95):.0f}')
axes[0, 0].set_xlabel('Text Length (tokens)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Input Text Length Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Summary length distribution
axes[0, 1].hist(df['summary_tokens'], bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].axvline(df['summary_tokens'].median(), color='red', linestyle='--', linewidth=2, 
                   label=f'Median: {df["summary_tokens"].median():.0f}')
axes[0, 1].axvline(df['summary_tokens'].quantile(0.95), color='orange', linestyle='--', linewidth=2, 
                   label=f'95th: {df["summary_tokens"].quantile(0.95):.0f}')
axes[0, 1].set_xlabel('Summary Length (tokens)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Output Summary Length Distribution', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Scatter plot - Input vs Output
axes[0, 2].scatter(df['text_tokens'], df['summary_tokens'], alpha=0.3, s=20)
axes[0, 2].set_xlabel('Text Length (tokens)', fontsize=11)
axes[0, 2].set_ylabel('Summary Length (tokens)', fontsize=11)
axes[0, 2].set_title('Input vs Output Length Relationship', fontweight='bold')
axes[0, 2].grid(True, alpha=0.3)

# 4. Box plots comparison
bp = axes[1, 0].boxplot([df['text_tokens'], df['summary_tokens']], 
                         labels=['Text (Input)', 'Summary (Output)'],
                         patch_artist=True)
bp['boxes'][0].set_facecolor('skyblue')
bp['boxes'][1].set_facecolor('lightgreen')
axes[1, 0].set_ylabel('Token Count', fontsize=11)
axes[1, 0].set_title('Length Distribution Comparison', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# 5. Compression ratio distribution
axes[1, 1].hist(compression_ratio, bins=50, edgecolor='black', alpha=0.7, color='plum')
axes[1, 1].axvline(compression_ratio.median(), color='red', linestyle='--', linewidth=2, 
                   label=f'Median: {compression_ratio.median():.2f}x')
axes[1, 1].set_xlabel('Compression Ratio', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Text-to-Summary Compression Ratio', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

# 6. Cumulative distribution
sorted_text = np.sort(df['text_tokens'])
cumulative = np.arange(1, len(sorted_text) + 1) / len(sorted_text) * 100
axes[1, 2].plot(sorted_text, cumulative, linewidth=2)
axes[1, 2].axhline(95, color='orange', linestyle='--', linewidth=2, label='95%')
axes[1, 2].axhline(99, color='red', linestyle='--', linewidth=2, label='99%')
axes[1, 2].set_xlabel('Text Length (tokens)', fontsize=11)
axes[1, 2].set_ylabel('Cumulative Percentage', fontsize=11)
axes[1, 2].set_title('Cumulative Distribution of Input Lengths', fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('length_analysis_visualization.png', dpi=300, bbox_inches='tight')
print("\n✓ Visualization saved to 'length_analysis_visualization.png'")
plt.show()

# ============================================================================
# TRUNCATION COVERAGE VISUALIZATION
# ============================================================================

plt.figure(figsize=(10, 6))
plt.plot(truncation_df['max_length'], truncation_df['coverage'], marker='o', linewidth=2, markersize=8)
plt.axhline(y=95, color='orange', linestyle='--', label='95% coverage', alpha=0.7)
plt.axhline(y=99, color='red', linestyle='--', label='99% coverage', alpha=0.7)
plt.xlabel('Maximum Token Length', fontsize=12)
plt.ylabel('Percentage of Examples Covered', fontsize=12)
plt.title('Input Truncation Coverage Analysis', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('truncation_coverage.png', dpi=300, bbox_inches='tight')
print("✓ Truncation coverage plot saved to 'truncation_coverage.png'")
plt.show()

# ============================================================================
# EXPORT SUMMARY STATISTICS
# ============================================================================

summary_stats = pd.DataFrame({
    'Metric': [
        'Input: Mean tokens',
        'Input: Median tokens',
        'Input: 95th percentile',
        'Input: 99th percentile',
        'Input: Max tokens',
        'Output: Mean tokens',
        'Output: Median tokens',
        'Output: 95th percentile',
        'Output: 99th percentile',
        'Output: Max tokens',
        'Compression Ratio: Mean',
        'Compression Ratio: Median',
        'Recommended max_tokens',
        'Recommended context window'
    ],
    'Value': [
        df['text_tokens'].mean(),
        df['text_tokens'].median(),
        df['text_tokens'].quantile(0.95),
        df['text_tokens'].quantile(0.99),
        df['text_tokens'].max(),
        df['summary_tokens'].mean(),
        df['summary_tokens'].median(),
        df['summary_tokens'].quantile(0.95),
        df['summary_tokens'].quantile(0.99),
        df['summary_tokens'].max(),
        compression_ratio.mean(),
        compression_ratio.median(),
        int(df['summary_tokens'].quantile(0.95) * 1.3),
        int(df['text_tokens'].quantile(0.99))
    ]
})

print("\n\n📊 SUMMARY STATISTICS TABLE:")
print("="*50)
print(summary_stats.to_string(index=False))

summary_stats.to_csv('length_statistics.csv', index=False)
print("\n✓ Statistics saved to 'length_statistics.csv'")

# ============================================================================
# FINAL RECOMMENDATIONS
# ============================================================================

print("\n\n" + "="*70)
print("FINAL RECOMMENDATIONS FOR FINE-TUNING")
print("="*70)

recommended_max = int(df['summary_tokens'].quantile(0.95) * 1.3)
context_99 = int(df['text_tokens'].quantile(0.99))

print(f"\n1. 🎯 max_tokens parameter: {recommended_max}")
print(f"   - This covers 95% of your summaries with a 30% safety buffer")
print(f"   - Alternative (99th): {int(df['summary_tokens'].quantile(0.99) * 1.2)}")

print(f"\n2. 📏 Context window requirement: {context_99} tokens")
print(f"   - This handles 99% of your input texts")
print(f"   - Consider truncating longer inputs or using a model with sufficient context")

print(f"\n3. ✂️  Input truncation strategy:")
for length in [512, 1024, 2048, 4096]:
    pct = (df['text_tokens'] <= length).mean() * 100
    if pct >= 95:
        print(f"   → Truncate at {length} tokens (covers {pct:.1f}%)")
        break
else:
    print(f"   → Consider {context_99} tokens or higher")

print(f"\n4. 📊 Dataset info:")
print(f"   - Total examples: {len(df):,}")
print(f"   - Avg tokens per example: {df['total_tokens'].mean():.0f}")
print(f"   - Compression ratio: {compression_ratio.mean():.1f}x")

print(f"\n5. 💡 Tips:")
print(f"   - Monitor validation loss to avoid overfitting")
print(f"   - Consider splitting very long inputs if they exceed model limits")
print(f"   - Use early stopping based on validation performance")

print("\n" + "="*70)
print("\n✓ Analysis complete!")

In [ ]:
import pandas as pd

In [2]:
import pandas as pd
from datasets import Dataset

# =========================
# Utility: Clean SFT dataframe
# =========================
def clean_sft_df(df):
    return df[
        df["text"].apply(lambda x: isinstance(x, str) and x.strip() != "") &
        df["summary"].apply(lambda x: isinstance(x, str) and x.strip() != "")
    ].reset_index(drop=True)


# =========================
# Utility: Clean DPO dataframe
# =========================
def clean_dpo_df(df):
    return df[
        df["text"].apply(lambda x: isinstance(x, str) and x.strip() != "") &
        df["summary"].apply(lambda x: isinstance(x, str) and x.strip() != "") &
        df["generated_summary"].apply(lambda x: isinstance(x, str) and x.strip() != "")
    ].reset_index(drop=True)


In [ ]:
# import pandas as pd
# from datasets import Dataset

# # =========================
# # -------- LOAD & SPLIT ---
# # =========================
# # Load single CSV
# df = pd.read_json("/kaggle/input/datasets/abhisheksaha1234/ami-and-qmsum/ami_summary.jsonl", lines=True)
# print(f"Total dataset size: {len(df)}")

# # Shuffle the entire dataset first
# df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# # Split into SFT and DPO portions
# # Assuming you want 30k for SFT and rest for DPO
# sft_size = 100
# df_sft = df.iloc[:sft_size].copy()
# df_dpo = df.iloc[sft_size:].copy()

# print(f"SFT portion: {len(df_sft)}")
# print(f"DPO portion: {len(df_dpo)}")

# # =========================
# # -------- SFT DATA -------
# # =========================
# df_clean = clean_dpo_df(df_sft) #Exception for SCITLDR as same dataframe for both 
# # Already split, so no need to sample again - just use all cleaned data
# print("\nSFT DATASET")
# print(f"Original size: {len(df_sft)}")
# print(f"Cleaned size: {len(df_clean)}")
# print(f"Dropped rows : {len(df_sft) - len(df_clean)}")
# sft_dataset = Dataset.from_pandas(df_clean)

# # =========================
# # -------- DPO DATA -------
# # =========================
# df_clean = clean_dpo_df(df_dpo)
# print("\nDPO DATASET (Before split)")
# print(f"Original size: {len(df_dpo)}")
# print(f"Cleaned size : {len(df_clean)}")
# print(f"Dropped rows : {len(df_dpo) - len(df_clean)}")
# dataset = Dataset.from_pandas(df_clean)

# # 90% DPO train, 10% test
# split = dataset.train_test_split(test_size=0.1, seed=42)
# dpo_dataset  = split["train"]
# test_dataset = split["test"]

# # =========================
# # -------- SUMMARY --------
# # =========================
# print("\n" + "="*60)
# print("FINAL DATASET SPLIT SUMMARY")
# print("="*60)
# print(f"SFT training samples : {len(sft_dataset)}")
# print(f"DPO training samples : {len(dpo_dataset)}")
# print(f"Test samples         : {len(test_dataset)}")
# print("="*60)

In [4]:
import pandas as pd
from datasets import Dataset

# =========================
# -------- LOAD & SPLIT ---
# =========================

# Load dataset
df = pd.read_json(
    "/kaggle/input/datasets/abhisheksaha1234/ami-and-qmsum/ami_summary.jsonl",
    lines=True
)

print(f"Total dataset size: {len(df)}")

# Shuffle entire dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# -------------------------
# SFT portion (100 samples)
# -------------------------
sft_size = 100
df_sft = df.iloc[:sft_size].copy()

# Remaining used only for testing
df_test = df.iloc[sft_size:].copy()

print(f"SFT portion : {len(df_sft)}")
print(f"Test portion: {len(df_test)}")

# =========================
# -------- SFT DATA -------
# =========================

df_sft_clean = clean_sft_df(df_sft)

print("\nSFT DATASET")
print(f"Original size: {len(df_sft)}")
print(f"Cleaned size : {len(df_sft_clean)}")
print(f"Dropped rows : {len(df_sft) - len(df_sft_clean)}")

sft_dataset = Dataset.from_pandas(df_sft_clean)

# =========================
# -------- TEST DATA ------
# =========================

df_test_clean = clean_sft_df(df_test)

print("\nTEST DATASET")
print(f"Original size: {len(df_test)}")
print(f"Cleaned size : {len(df_test_clean)}")
print(f"Dropped rows : {len(df_test) - len(df_test_clean)}")

test_dataset = Dataset.from_pandas(df_test_clean)

# =========================
# -------- SUMMARY --------
# =========================

print("\n" + "="*60)
print("FINAL DATASET SPLIT SUMMARY")
print("="*60)
print(f"SFT training samples : {len(sft_dataset)}")
print(f"Test samples         : {len(test_dataset)}")
print("="*60)


Total dataset size: 114
SFT portion : 100
Test portion: 14

SFT DATASET
Original size: 100
Cleaned size : 100
Dropped rows : 0

TEST DATASET
Original size: 14
Cleaned size : 14
Dropped rows : 0

FINAL DATASET SPLIT SUMMARY
SFT training samples : 100
Test samples         : 14


In [ ]:
# =========================
# -------- SFT DATA -------
# =========================
# df = pd.read_csv("subset_5000_33000.csv")   # SFT CSV
df_clean = clean_sft_df(df)

# Shuffle and take 30k
df_clean = df_clean.sample(n=30000, random_state=42).reset_index(drop=True)

print("SFT DATASET")
print(f"Original size: {len(df)}")
print(f"Cleaned size (sampled): {len(df_clean)}")
print(f"Dropped rows : {len(df) - len(df_clean)}")

sft_dataset = Dataset.from_pandas(df_clean)



# =========================
# -------- DPO DATA -------
# =========================
df = pd.read_csv("/kaggle/input/cnndailydpo-sft/data 4.csv")      # DPO CSV
df_clean = clean_dpo_df(df)

print("\nDPO DATASET (Before split)")
print(f"Original size: {len(df)}")
print(f"Cleaned size : {len(df_clean)}")
print(f"Dropped rows : {len(df) - len(df_clean)}")

dataset = Dataset.from_pandas(df_clean)

# 90% DPO train, 10% test
split = dataset.train_test_split(test_size=0.1, seed=42)

dpo_dataset  = split["train"]
test_dataset = split["test"]


# =========================
# -------- SUMMARY --------
# =========================
print("\n" + "="*60)
print("FINAL DATASET SPLIT SUMMARY")
print("="*60)
print(f"SFT training samples : {len(sft_dataset)}")
print(f"DPO training samples : {len(dpo_dataset)}")
print(f"Test samples         : {len(test_dataset)}")
print("="*60)


## 4. Prepare SFT Dataset

In [5]:
def prepare_sft_format(examples):
    
    """Format data for supervised fine-tuning (Summary writing)"""
    messages_list = []
    for text, summary in zip(examples["text"], examples["summary"]):
        # ALIGNED with your actual data format
        user_prompt = (
            "Summarize the following document concisely.\n\n"
            "Requirements:\n"
            "- Capture the main points in 5-6 sentences\n"
            "- Focus on key facts and findings\n"
            "- Use clear, engaging language\n"
            "- Avoid unnecessary details\n\n"
            f"Document:\n{text[:8192]}"
        )

        
        messages = [
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": summary.strip()},
        ]
        messages_list.append(messages)
    return {"messages": messages_list}



# Apply formatting
sft_formatted = sft_dataset.map(
    prepare_sft_format,
    batched=True,
    remove_columns=["text", "summary"],
    desc="Formatting SFT data",
)

print(f"\n✅ SFT dataset prepared: {len(sft_formatted)} samples")
print("\nSample:")
print(sft_formatted[0]['messages'])

Formatting SFT data:   0%|          | 0/100 [00:00<?, ? examples/s]


✅ SFT dataset prepared: 100 samples

Sample:
[{'content': "Summarize the following document concisely.\n\nRequirements:\n- Capture the main points in 5-6 sentences\n- Focus on key facts and findings\n- Use clear, engaging language\n- Avoid unnecessary details\n\nDocument:\nSpeaker A: Hi everyone, hope you had a nice lunch. Um. Alright we're moving on to conceptual design. Um, I'll just review what we did in our last meeting. Um, under marketing we targeted our audience, and Um, yeah. That was generally how helpful that was. Um, then we considered some design options with how it should look, um, we discussed an iPod-like button system which, uh, we haven't concluded but we're Right, um So, if you all have presentations to do, we can see what where you've come from our last time. Does everyone have presentations? Okay. Would anybody like to go first? Okay. And why not wood? And why not wood? 'S good. Hmm. Mm. I think it's the next it's the blue one, yeah. Yeah but you're not gonna wear 

In [6]:
# Apply chat template for SFT
def apply_sft_chat_template(example, tokenizer):
    """Apply chat template to messages"""
    messages = example["messages"]
    
    # Add empty system message if needed
    if messages[0]["role"] != "system":
        messages.insert(0, {"role": "system", "content": ""})
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    
    return {"text": text}

sft_formatted = sft_formatted.map(
    lambda x: apply_sft_chat_template(x, base_tokenizer),
    remove_columns=["messages"],
    desc="Applying chat template",
)

print("\n✅ Chat template applied")
print("\nSample formatted text:")
print(sft_formatted[0]['text'][:5000] + "...")

Applying chat template:   0%|          | 0/100 [00:00<?, ? examples/s]

NameError: name 'base_tokenizer' is not defined

## 5. Stage 1: Supervised Fine-Tuning (SFT)

In [ ]:
!pip install -U peft


In [ ]:
# Prepare model for SFT
sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3-8B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    # load_in_8bit=False,
    # dtype=torch.bfloat16,
)

# Set chat template
if sft_tokenizer.chat_template is None:
    sft_tokenizer.chat_template = base_tokenizer.chat_template

# Add LoRA adapters
sft_model = FastLanguageModel.get_peft_model(
    sft_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("\n✅ SFT model prepared with LoRA adapters")

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_trainer = SFTTrainer(
    model=sft_model,
    tokenizer=sft_tokenizer,
    train_dataset=sft_formatted,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="sft_outputs",
        report_to="none",
        
        # NEW: HIGHLY RECOMMENDED
        save_strategy="steps",
        save_steps=500,
        # Reasoning: Save checkpoints periodically. If training crashes,
        # you don't lose everything.
    ),
)

print("\n" + "="*60)
print("STARTING SFT TRAINING")
print("="*60)

In [ ]:
# Train SFT model
sft_stats = sft_trainer.train()

print("\n" + "="*60)
print("SFT TRAINING COMPLETED")
print("="*60)
print(f"Training time: {sft_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training time: {sft_stats.metrics['train_runtime']/60:.2f} minutes")

In [ ]:
# Save SFT model
sft_model.save_pretrained("llama3_spotlight_sft")
sft_tokenizer.save_pretrained("llama3_spotlight_sft")

print("\n✅ SFT model saved to 'llama3_spotlight_sft'")

## 5.1 Loading from hugging face

In [ ]:
from huggingface_hub import login

login(token="")


In [ ]:
import unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Abhishekkk3/llama3-spotlight-sft",
    max_seq_length=4096,
    load_in_4bit=False,
    load_in_8bit=False,
    dtype=torch.bfloat16,  
)


In [ ]:
if sft_tokenizer.chat_template is None:
    sft_tokenizer.chat_template = "{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\\n\\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}{% endif %}"

## 6. Prepare DPO Dataset

In [ ]:
import re
from typing import Literal

def apply_chat_template(
    example,
    tokenizer,
    task: Literal["sft", "generation", "rm", "dpo"] = "sft",
    assistant_prefix="<|assistant|>\n",
):
    def _strip_prefix(s, pattern):
        return re.sub(f"^{re.escape(pattern)}", "", s)

    if task == "dpo":
        if all(k in example.keys() for k in ("chosen", "rejected")):
            # Extract prompt messages
            prompt_messages = [
                [msg for msg in example["chosen"] if msg["role"] == "user"][0]
            ]
            
            # Insert system message if needed
            if example["chosen"][0]["role"] != "system":
                prompt_messages.insert(0, {"role": "system", "content": ""})
            else:
                prompt_messages.insert(0, example["chosen"][0])
            
            chosen_messages = example["chosen"][1:]
            rejected_messages = example["rejected"][1:]
            
            example["text_chosen"] = tokenizer.apply_chat_template(
                chosen_messages, tokenize=False
            )
            example["text_rejected"] = tokenizer.apply_chat_template(
                rejected_messages, tokenize=False
            )
            example["text_prompt"] = tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True
            )
            
            example["text_chosen"] = _strip_prefix(
                example["text_chosen"], assistant_prefix
            )
            example["text_rejected"] = _strip_prefix(
                example["text_rejected"], assistant_prefix
            )
        else:
            raise ValueError(
                f"Could not format example as dialogue for `dpo` task! Require `[chosen, rejected]` keys but found {list(example.keys())}"
            )
    
    return example

In [ ]:
def prepare_dpo_format(examples):
    """Prepare data for DPO training - CONSISTENT with SFT"""
    chosen_messages = []
    rejected_messages = []
    
    for text, summary, generated_summary in zip(
        examples["text"],
        examples["summary"],
        examples["generated_summary"]
    ):
        generated_summary_cleaned = generated_summary.replace("[SUMMARY]", "").strip()
        
        # Match SFT prompt format exactly
        user_prompt = (
            "Summarize the following document concisely.\n\n"
            "Requirements:\n"
            "- Capture main points in 2-4 sentences\n"
            "- Focus on key facts and findings\n"
            "- Use clear, engaging language\n"
            "- Avoid unnecessary details\n\n"
            f"Document:\n{text[:2048]}"  # Truncate like SFT
        )
        
        # Chosen: human-written summary (better)
        chosen_messages.append([
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": summary.strip()},
        ])
        
        # Rejected: generated summary (worse)
        rejected_messages.append([
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": generated_summary_cleaned},
        ])
    
    return {
        "chosen": chosen_messages,
        "rejected": rejected_messages,
    }

# Apply DPO formatting
dpo_formatted = dpo_dataset.map(
    prepare_dpo_format,
    batched=True,
    num_proc=4,
    remove_columns=["text", "summary", "generated_summary"],
    desc="Preparing DPO format",
)

print(f"\n✅ DPO dataset prepared: {len(dpo_formatted)} samples")

In [ ]:
# Apply chat template for DPO
column_names = list(dpo_formatted.features)
dpo_formatted = dpo_formatted.map(
    apply_chat_template,
    fn_kwargs={"tokenizer": sft_tokenizer, "task": "dpo"},
    num_proc=4,
    remove_columns=column_names,
    desc="Applying chat template for DPO",
)

# Rename columns
dpo_formatted = dpo_formatted.rename_columns(
    {
        "text_prompt": "prompt",
        "text_chosen": "chosen",
        "text_rejected": "rejected",
    }
)

print("\n✅ DPO dataset formatted")
print("\nSample DPO data:")
print(f"Prompt length: {len(dpo_formatted[0]['prompt'])}")
print(f"Chosen length: {len(dpo_formatted[0]['chosen'])}")
print(f"Rejected length: {len(dpo_formatted[0]['rejected'])}")

## 7. Stage 2: Direct Preference Optimization (DPO)

In [ ]:
# Load the SFT model for DPO training
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name="llama3_spotlight_sft",
    # model_name="Abhishekkk3/llama3-spotlight-sft",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None
    # load_in_8bit=False,
    # dtype=torch.bfloat16,
)

# Set chat template
if dpo_tokenizer.chat_template is None:
    dpo_tokenizer.chat_template = sft_tokenizer.chat_template

print("\n✅ SFT model loaded for DPO training")

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,  # Use implicit reference model
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        num_train_epochs=2,
        learning_rate=5e-6,
        # ✅ Checkpointing
        save_strategy="steps",
        save_steps=20,
        # save_total_limit=3,   # keeps only last 3 checkpoints (recommended)
        max_steps=200, 
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="dpo_outputs",
        report_to="none",
    ),
    beta=0.05,
    train_dataset=dpo_formatted,
    tokenizer=dpo_tokenizer,
    max_length=2560,
    max_prompt_length=2048,
)

print("\n" + "="*60)
print("STARTING DPO TRAINING")
print("="*60)

In [ ]:
# Train DPO model
dpo_stats = dpo_trainer.train()

print("\n" + "="*60)
print("DPO TRAINING COMPLETED")
print("="*60)
print(f"Training time: {dpo_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training time: {dpo_stats.metrics['train_runtime']/60:.2f} minutes")

In [ ]:
# Save DPO model
dpo_model.save_pretrained("llama3_spotlight_sft_dpo")
dpo_tokenizer.save_pretrained("llama3_spotlight_sft_dpo")

print("\n✅ SFT+DPO model saved to 'llama3_spotlight_sft_dpo'")

## 8. Prepare Test Dataset

In [ ]:
# Prepare test dataset in DPO format (to get prompts and references)
test_formatted = test_dataset.map(
    prepare_dpo_format,
    batched=True,
    num_proc=4,
    remove_columns=["text", "summary", "generated_summary"],
    desc="Preparing test data",
)

column_names = list(test_formatted.features)
test_formatted = test_formatted.map(
    apply_chat_template,
    fn_kwargs={"tokenizer": dpo_tokenizer, "task": "dpo"},
    num_proc=4,
    remove_columns=column_names,
    desc="Applying chat template for test",
)

test_formatted = test_formatted.rename_columns(
    {
        "text_prompt": "prompt",
        "text_chosen": "chosen",
        "text_rejected": "rejected",
    }
)

print(f"\n✅ Test dataset prepared: {len(test_formatted)} samples")

## 9. Model Comparison: Base vs SFT vs SFT+DPO

In [ ]:
# print("\n" + "="*80)
# print("LOADING ALL MODELS FOR COMPARISON")
# print("="*80)

# # 1. Load Base Model
# print("\n1. Loading Base Model...")
# base_model_eval, base_tokenizer_eval = FastLanguageModel.from_pretrained(
#     model_name="unsloth/Llama-3-8B",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# if base_tokenizer_eval.chat_template is None:
#     base_tokenizer_eval.chat_template = dpo_tokenizer.chat_template
# FastLanguageModel.for_inference(base_model_eval)
# print("✅ Base model loaded")

# # 2. Load SFT Model
# print("\n2. Loading SFT Model...")
# sft_model_eval, sft_tokenizer_eval = FastLanguageModel.from_pretrained(
#     model_name="llama3_spotlight_sft",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# FastLanguageModel.for_inference(sft_model_eval)
# print("✅ SFT model loaded")

# # 3. Load SFT+DPO Model
# print("\n3. Loading SFT+DPO Model...")
# sft_dpo_model_eval, sft_dpo_tokenizer_eval = FastLanguageModel.from_pretrained(
#     model_name="llama3_spotlight_sft_dpo",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# FastLanguageModel.for_inference(sft_dpo_model_eval)
# print("✅ SFT+DPO model loaded")

# print("\n" + "="*80)
# print("ALL MODELS LOADED AND READY FOR EVALUATION")
# print("="*80)

In [ ]:
# from tqdm import tqdm

# def generate_spotlight(model, tokenizer, prompt_text, max_new_tokens=256):
#     """Generate spotlight given the formatted prompt"""
#     inputs = tokenizer(
#         prompt_text,
#         return_tensors="pt",
#         truncation=True,
#         max_length=1024,
#     ).to(model.device)
    
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#             eos_token_id=tokenizer.eos_token_id,
#         )
    
#     generated_text = tokenizer.decode(
#         outputs[0][inputs['input_ids'].shape[1]:],
#         skip_special_tokens=True
#     )
    
#     return generated_text.strip()

In [ ]:
# # Test tokenizer independently
# test_text = "Hello, this is a test."
# test_tokens = sft_tokenizer.encode(test_text)
# decoded = sft_tokenizer.decode(test_tokens)
# print(f"Original: {test_text}")
# print(f"Decoded: {decoded}")
# # Should match!

In [ ]:
# print("\n" + "="*80)
# print("RUNNING INFERENCE ON TEST SET")
# print("="*80)

# num_samples = len(test_formatted)
# print(f"\nEvaluating on {num_samples} test samples...\n")

# base_predictions = []
# sft_predictions = []
# sft_dpo_predictions = []
# references = []

# for i in tqdm(range(num_samples), desc="Generating predictions"):
#     example = test_formatted[i]
#     prompt_text = example["prompt"]
#     ref_text = example["chosen"]
    
#     # Generate with base model
#     base_pred = generate_spotlight(base_model_eval, base_tokenizer_eval, prompt_text)
    
#     # Generate with SFT model
#     sft_pred = generate_spotlight(sft_model_eval, sft_tokenizer_eval, prompt_text)
    
#     # Generate with SFT+DPO model
#     sft_dpo_pred = generate_spotlight(sft_dpo_model_eval, sft_dpo_tokenizer_eval, prompt_text)
    
#     references.append(ref_text)
#     base_predictions.append(base_pred)
#     sft_predictions.append(sft_pred)
#     sft_dpo_predictions.append(sft_dpo_pred)

# print(f"\n✅ Generated {len(base_predictions)} predictions for each model!")

## 9.1 BP16 model take larger space so prev code cannot execute here.

In [ ]:
import torch
import gc
from tqdm import tqdm
from unsloth import FastLanguageModel

In [ ]:
def build_prompt(tokenizer, text):
    """Build prompt matching SFT training format"""
    
    # Match your SFT training: empty system message
    system_prompt = ""
    
    # Match your SFT user prompt EXACTLY
    user_prompt = (
        "Summarize the following document concisely.\n\n"
        "Requirements:\n"
        "- Capture main points in 2-4 sentences\n"
        "- Focus on key facts and findings\n"
        "- Use clear, engaging language\n"
        "- Avoid unnecessary details\n\n"
        f"Document:\n{text[:2048]}"  # Truncate at 2048 like training
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


In [ ]:
def generate_spotlight(model, tokenizer, prompt_text, max_new_tokens=160):
    """Generate summary given the formatted prompt"""
    
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=2560,  # Match training max_seq_length
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # 160 tokens (99th percentile)
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,  # Prevent repetition
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the new tokens (excluding prompt)
    generated_text = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],  # Slice off prompt
        skip_special_tokens=True
    )
    
    return generated_text.strip()

In [ ]:
sft_formatted[0]

In [ ]:
# Take first 10 rows
subset = test_formatted.select(range(200))

# Extract chosen responses
references = [example["chosen"] for example in subset]

print(len(references))  # should print 10
print(references[0])    # check first one


In [ ]:
print("\n" + "="*80)
print("RUNNING INFERENCE ON TEST SET")
print("="*80)

# Define max_seq_length if not already defined
# max_seq_length = 2048

num_samples = min(200, len(test_formatted))
print(f"\nEvaluating on {num_samples} test samples...\n")

# Initialize prediction lists
base_predictions = []
sft_predictions = []
sft_dpo_predictions = []
# references = []

In [ ]:
print("\nLoading Base Model...")
base_model_eval, base_tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3-8B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    # load_in_8bit=False,
    dtype=None
)
FastLanguageModel.for_inference(base_model_eval)

for i in tqdm(range(num_samples), desc="Base Model"):
    example = test_formatted[i]
    prompt_text = example["prompt"]
    pred = generate_spotlight(base_model_eval, base_tokenizer_eval, prompt_text)
    # print(pred)
    base_predictions.append(pred)

# Clean up
del base_model_eval
del base_tokenizer_eval
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("✅ Base model evaluation complete.")

In [ ]:
# Print one example to see the structure
print("Example keys:", test_formatted[0].keys())
print("\nPrompt field:", test_formatted[0]["prompt"][:500])
if "text" in test_formatted[0]:
    print("\nText field:", test_formatted[0]["text"][:500])

In [ ]:
print("\nLoading SFT Model...")
sft_model_eval, sft_tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/llama3_spotlight_sft",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    # load_in_8bit=False,
    # dtype=torch.bfloat16
)
FastLanguageModel.for_inference(sft_model_eval)

for i in tqdm(range(num_samples), desc="SFT Model"):
    example = test_formatted[i]
    prompt_text = example["prompt"]  # ✅ Already formatted - use directly!
    pred = generate_spotlight(sft_model_eval, sft_tokenizer_eval, prompt_text)
    # print(pred)
    sft_predictions.append(pred)

# Clean up
del sft_model_eval
del sft_tokenizer_eval
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("✅ SFT model evaluation complete.")

In [ ]:
print("\nLoading SFT+DPO Model...")
sft_dpo_model_eval, sft_dpo_tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/dpo_outputs/checkpoint-120",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    # load_in_8bit=False,
    dtype=None
)
FastLanguageModel.for_inference(sft_dpo_model_eval)

for i in tqdm(range(num_samples), desc="SFT+DPO Model"):
    example = test_formatted[i]
    prompt_text = example["prompt"]
    pred = generate_spotlight(sft_dpo_model_eval, sft_dpo_tokenizer_eval, prompt_text)
    sft_dpo_predictions.append(pred)

# Clean up
del sft_dpo_model_eval
del sft_dpo_tokenizer_eval
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("✅ SFT+DPO model evaluation complete.")

In [ ]:
references[0]

In [ ]:


base_predictions[0]

In [ ]:
sft_dpo_predictions[0]

In [ ]:
sft_predictions[0]

## 10. Calculate Metrics

In [ ]:
print("\n" + "="*80)
print("CALCULATING ROUGE SCORES")
print("="*80)

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_rouge_scores(predictions, references):
    """Calculate average ROUGE scores"""
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    return {
        'rouge1': sum(rouge1_scores) / len(rouge1_scores),
        'rouge2': sum(rouge2_scores) / len(rouge2_scores),
        'rougeL': sum(rougeL_scores) / len(rougeL_scores),
    }

base_rouge = calculate_rouge_scores(base_predictions, references)
sft_rouge = calculate_rouge_scores(sft_predictions, references)
sft_dpo_rouge = calculate_rouge_scores(sft_dpo_predictions, references)

print("\n📊 ROUGE SCORES COMPARISON:")
print("-" * 80)
print(f"{'Metric':<15} {'Base':<15} {'SFT':<15} {'SFT+DPO':<15}")
print("-" * 80)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    print(f"{metric:<15} {base_rouge[metric]:<15.4f} {sft_rouge[metric]:<15.4f} {sft_dpo_rouge[metric]:<15.4f}")
print("-" * 80)

print("\n📈 IMPROVEMENT OVER BASE:")
print("-" * 80)
print(f"{'Metric':<15} {'SFT Improvement':<20} {'SFT+DPO Improvement':<25}")
print("-" * 80)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    sft_imp = ((sft_rouge[metric] - base_rouge[metric]) / base_rouge[metric]) * 100
    dpo_imp = ((sft_dpo_rouge[metric] - base_rouge[metric]) / base_rouge[metric]) * 100
    print(f"{metric:<15} {sft_imp:+.2f}%{'':<15} {dpo_imp:+.2f}%")
print("-" * 80)

In [ ]:
print("\n" + "="*80)
print("CALCULATING ROUGE SCORES")
print("="*80)

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_rouge_scores(predictions, references, exclude_zeros=False):
    """
    Calculate average ROUGE scores
    
    Parameters:
    -----------
    predictions : list
        List of predicted summaries
    references : list
        List of reference summaries
    exclude_zeros : bool
        If True, exclude zero scores from average calculation
    
    Returns:
    --------
    dict : Average ROUGE scores and counts
    """
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    if exclude_zeros:
        # Filter out zeros and calculate average on non-zero values only
        rouge1_nonzero = [s for s in rouge1_scores if s > 0]
        rouge2_nonzero = [s for s in rouge2_scores if s > 0]
        rougeL_nonzero = [s for s in rougeL_scores if s > 0]
        
        return {
            'rouge1': sum(rouge1_nonzero) / len(rouge1_nonzero) if rouge1_nonzero else 0.0,
            'rouge2': sum(rouge2_nonzero) / len(rouge2_nonzero) if rouge2_nonzero else 0.0,
            'rougeL': sum(rougeL_nonzero) / len(rougeL_nonzero) if rougeL_nonzero else 0.0,
            'rouge1_count': len(rouge1_nonzero),
            'rouge2_count': len(rouge2_nonzero),
            'rougeL_count': len(rougeL_nonzero),
            'rouge1_zeros': len(rouge1_scores) - len(rouge1_nonzero),
            'rouge2_zeros': len(rouge2_scores) - len(rouge2_nonzero),
            'rougeL_zeros': len(rougeL_scores) - len(rougeL_nonzero),
        }
    else:
        return {
            'rouge1': sum(rouge1_scores) / len(rouge1_scores),
            'rouge2': sum(rouge2_scores) / len(rouge2_scores),
            'rougeL': sum(rougeL_scores) / len(rougeL_scores),
        }

# Calculate scores
base_rouge = calculate_rouge_scores(base_predictions, references, exclude_zeros=False)
sft_rouge = calculate_rouge_scores(sft_predictions, references, exclude_zeros=True)  # Exclude zeros for SFT
sft_dpo_rouge = calculate_rouge_scores(sft_dpo_predictions, references, exclude_zeros=False)

print("\n📊 ROUGE SCORES COMPARISON:")
print("-" * 80)
print(f"{'Metric':<15} {'Base':<15} {'SFT':<15} {'SFT+DPO':<15}")
print("-" * 80)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    print(f"{metric:<15} {base_rouge[metric]:<15.4f} {sft_rouge[metric]:<15.4f} {sft_dpo_rouge[metric]:<15.4f}")
print("-" * 80)

# Show SFT statistics (zeros excluded)
if 'rouge1_count' in sft_rouge:
    print("\n⚠️  SFT Statistics (zeros excluded from average):")
    print("-" * 80)
    total_samples = len(references)
    print(f"{'Metric':<15} {'Non-zero samples':<20} {'Zero samples':<20} {'Coverage':<15}")
    print("-" * 80)
    for metric in ['rouge1', 'rouge2', 'rougeL']:
        nonzero = sft_rouge[f'{metric}_count']
        zeros = sft_rouge[f'{metric}_zeros']
        coverage = (nonzero / total_samples) * 100
        print(f"{metric:<15} {nonzero:<20} {zeros:<20} {coverage:.1f}%")
    print("-" * 80)

print("\n📈 IMPROVEMENT OVER BASE:")
print("-" * 80)
print(f"{'Metric':<15} {'SFT Improvement':<20} {'SFT+DPO Improvement':<25}")
print("-" * 80)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    sft_imp = ((sft_rouge[metric] - base_rouge[metric]) / base_rouge[metric]) * 100
    dpo_imp = ((sft_dpo_rouge[metric] - base_rouge[metric]) / base_rouge[metric]) * 100
    print(f"{metric:<15} {sft_imp:+.2f}%{'':<15} {dpo_imp:+.2f}%")
print("-" * 80)

# Additional warning if many zeros
if 'rouge1_zeros' in sft_rouge:
    total = len(references)
    zero_pct = (sft_rouge['rouge1_zeros'] / total) * 100
    
    if zero_pct > 10:
        print(f"\n⚠️  WARNING: SFT model has {sft_rouge['rouge1_zeros']} samples ({zero_pct:.1f}%) with zero ROUGE-1 scores!")
        print(f"   This suggests the model may be:")
        print(f"   - Generating empty outputs")
        print(f"   - Severely undertrained")
        print(f"   - Producing irrelevant summaries")
        print(f"   - Consider inspecting failed samples")

In [ ]:
# ============================================================================
# INDIVIDUAL SCORES FOR FIRST 20 SAMPLES
# ============================================================================

print("\n" + "="*80)
print("INDIVIDUAL ROUGE SCORES - FIRST 20 SAMPLES")
print("="*80)

num_samples = min(20, len(references))

for i in range(num_samples):
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    
    # Calculate scores for this sample
    base_scores = scorer.score(references[i], base_predictions[i])
    sft_scores = scorer.score(references[i], sft_predictions[i])
    sft_dpo_scores = scorer.score(references[i], sft_dpo_predictions[i])
    
    # Print reference and predictions
    print(f"\n📝 Reference:\n{references[i]}\n")
    print(f"🤖 Base Prediction:\n{base_predictions[i]}\n")
    print(f"🎯 SFT Prediction:\n{sft_predictions[i]}\n")
    print(f"💎 SFT+DPO Prediction:\n{sft_dpo_predictions[i]}\n")
    
    # Print scores
    print(f"{'─'*80}")
    print(f"{'Metric':<12} {'Base':<18} {'SFT':<18} {'SFT+DPO':<18}")
    print(f"{'─'*80}")
    
    for metric in ['rouge1', 'rouge2', 'rougeL']:
        base_f = base_scores[metric].fmeasure
        sft_f = sft_scores[metric].fmeasure
        dpo_f = sft_dpo_scores[metric].fmeasure
        
        print(f"{metric:<12} {base_f:<18.4f} {sft_f:<18.4f} {dpo_f:<18.4f}")
    
    # Show which model performed best
    avg_base = (base_scores['rouge1'].fmeasure + base_scores['rouge2'].fmeasure + base_scores['rougeL'].fmeasure) / 3
    avg_sft = (sft_scores['rouge1'].fmeasure + sft_scores['rouge2'].fmeasure + sft_scores['rougeL'].fmeasure) / 3
    avg_dpo = (sft_dpo_scores['rouge1'].fmeasure + sft_dpo_scores['rouge2'].fmeasure + sft_dpo_scores['rougeL'].fmeasure) / 3
    
    best_score = max(avg_base, avg_sft, avg_dpo)
    
    print(f"{'─'*80}")
    print(f"{'Average':<12} {avg_base:<18.4f} {avg_sft:<18.4f} {avg_dpo:<18.4f}")
    print(f"{'─'*80}")
    
    if avg_dpo == best_score:
        print(f"🏆 Best: SFT+DPO (Avg: {avg_dpo:.4f})")
    elif avg_sft == best_score:
        print(f"🏆 Best: SFT (Avg: {avg_sft:.4f})")
    else:
        print(f"🏆 Best: Base (Avg: {avg_base:.4f})")

In [ ]:
print("\n" + "="*80)
print("CALCULATING BERTSCORE")
print("="*80)

from bert_score import score as bert_score

print("\nCalculating BERTScore for base model...")
P_base, R_base, F1_base = bert_score(
    base_predictions, references,
    lang="en", verbose=False,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Calculating BERTScore for SFT model...")
P_sft, R_sft, F1_sft = bert_score(
    sft_predictions, references,
    lang="en", verbose=False,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Calculating BERTScore for SFT+DPO model...")
P_sft_dpo, R_sft_dpo, F1_sft_dpo = bert_score(
    sft_dpo_predictions, references,
    lang="en", verbose=False,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

base_bertscore = {
    'precision': P_base.mean().item(),
    'recall': R_base.mean().item(),
    'f1': F1_base.mean().item(),
}

sft_bertscore = {
    'precision': P_sft.mean().item(),
    'recall': R_sft.mean().item(),
    'f1': F1_sft.mean().item(),
}

sft_dpo_bertscore = {
    'precision': P_sft_dpo.mean().item(),
    'recall': R_sft_dpo.mean().item(),
    'f1': F1_sft_dpo.mean().item(),
}

print("\n📊 BERTSCORE COMPARISON:")
print("-" * 80)
print(f"{'Metric':<15} {'Base':<15} {'SFT':<15} {'SFT+DPO':<15}")
print("-" * 80)
for metric in ['precision', 'recall', 'f1']:
    print(f"{metric:<15} {base_bertscore[metric]:<15.4f} {sft_bertscore[metric]:<15.4f} {sft_dpo_bertscore[metric]:<15.4f}")
print("-" * 80)

print("\n📈 IMPROVEMENT OVER BASE:")
print("-" * 80)
print(f"{'Metric':<15} {'SFT Improvement':<20} {'SFT+DPO Improvement':<25}")
print("-" * 80)
for metric in ['precision', 'recall', 'f1']:
    sft_imp = ((sft_bertscore[metric] - base_bertscore[metric]) / base_bertscore[metric]) * 100
    dpo_imp = ((sft_dpo_bertscore[metric] - base_bertscore[metric]) / base_bertscore[metric]) * 100
    print(f"{metric:<15} {sft_imp:+.2f}%{'':<15} {dpo_imp:+.2f}%")
print("-" * 80)

## 11. Save Results & Create Summary

In [ ]:
import pandas as pd

# Create detailed results DataFrame
results_df = pd.DataFrame({
    'reference': references,
    'base_prediction': base_predictions,
    'sft_prediction': sft_predictions,
    'sft_dpo_prediction': sft_dpo_predictions,
})

results_df.to_csv('model_comparison_results.csv', index=False)
print("✅ Detailed results saved to 'model_comparison_results.csv'")

# Create summary DataFrame
summary_data = {
    'Metric': [
        'ROUGE-1', 'ROUGE-2', 'ROUGE-L',
        'BERTScore-Precision', 'BERTScore-Recall', 'BERTScore-F1'
    ],
    'Base Model': [
        base_rouge['rouge1'], base_rouge['rouge2'], base_rouge['rougeL'],
        base_bertscore['precision'], base_bertscore['recall'], base_bertscore['f1']
    ],
    'SFT Model': [
        sft_rouge['rouge1'], sft_rouge['rouge2'], sft_rouge['rougeL'],
        sft_bertscore['precision'], sft_bertscore['recall'], sft_bertscore['f1']
    ],
    'SFT+DPO Model': [
        sft_dpo_rouge['rouge1'], sft_dpo_rouge['rouge2'], sft_dpo_rouge['rougeL'],
        sft_dpo_bertscore['precision'], sft_dpo_bertscore['recall'], sft_dpo_bertscore['f1']
    ],
}

summary_df = pd.DataFrame(summary_data)
summary_df['SFT Improvement (%)'] = (
    (summary_df['SFT Model'] - summary_df['Base Model']) / summary_df['Base Model'] * 100
).round(2)
summary_df['SFT+DPO Improvement (%)'] = (
    (summary_df['SFT+DPO Model'] - summary_df['Base Model']) / summary_df['Base Model'] * 100
).round(2)

print("\n" + "="*100)
print("FINAL EVALUATION SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

summary_df.to_csv('model_comparison_summary.csv', index=False)
print("\n✅ Summary saved to 'model_comparison_summary.csv'")

## 12. Sample Predictions Comparison

In [ ]:
print("\n" + "="*100)
print("SAMPLE PREDICTIONS COMPARISON")
print("="*100)

num_samples_to_show = min(10, len(references))

for i in range(num_samples_to_show):
    print(f"\n{'='*100}")
    print(f"SAMPLE {i+1}")
    print("="*100)
    
    print(f"\n📝 REFERENCE (Human-written):")
    print(f"{references[i]}")
    
    print(f"\n🤖 BASE MODEL:")
    print(f"{base_predictions[i]}")
    
    print(f"\n🎯 SFT MODEL:")
    print(f"{sft_predictions[i]}")
    
    print(f"\n🏆 SFT+DPO MODEL:")
    print(f"{sft_dpo_predictions[i]}")
    print()

## 13. Optional: Push Models to Hugging Face Hub

In [ ]:
# Uncomment and fill in your details to push models to HF Hub

# YOUR_HF_USERNAME = "your_username"
# YOUR_HF_TOKEN = "your_token"

# # Push SFT model
# sft_model.push_to_hub(
#     f"{YOUR_HF_USERNAME}/llama3-spotlight-sft",
#     token=YOUR_HF_TOKEN,
# )
# sft_tokenizer.push_to_hub(
#     f"{YOUR_HF_USERNAME}/llama3-spotlight-sft",
#     token=YOUR_HF_TOKEN,
# )

# # Push SFT+DPO model
# dpo_model.push_to_hub(
#     f"{YOUR_HF_USERNAME}/llama3-spotlight-sft-dpo",
#     token=YOUR_HF_TOKEN,
# )
# dpo_tokenizer.push_to_hub(
#     f"{YOUR_HF_USERNAME}/llama3-spotlight-sft-dpo",
#     token=YOUR_HF_TOKEN,
# )

# print("✅ Models pushed to Hugging Face Hub!")

## 14. Training Summary

In [ ]:
print("\n" + "="*100)
print("COMPLETE PIPELINE SUMMARY")
print("="*100)

print("\n📊 DATASET SPLIT:")
print(f"  Total samples: {len(dataset)}")
print(f"  SFT training: {len(sft_dataset)} ({len(sft_dataset)/len(dataset)*100:.1f}%)")
print(f"  DPO training: {len(dpo_dataset)} ({len(dpo_dataset)/len(dataset)*100:.1f}%)")
print(f"  Test: {len(test_dataset)} ({len(test_dataset)/len(dataset)*100:.1f}%)")

print("\n⏱️ TRAINING TIME:")
print(f"  SFT: {sft_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"  DPO: {dpo_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"  Total: {(sft_stats.metrics['train_runtime'] + dpo_stats.metrics['train_runtime'])/60:.2f} minutes")

print("\n💾 SAVED MODELS:")
print("  1. llama3_spotlight_sft (SFT only)")
print("  2. llama3_spotlight_sft_dpo (SFT + DPO)")

print("\n📈 BEST PERFORMING MODEL:")
# Determine best model based on ROUGE-L F1
scores = {
    'Base': base_rouge['rougeL'],
    'SFT': sft_rouge['rougeL'],
    'SFT+DPO': sft_dpo_rouge['rougeL']
}
best_model = max(scores, key=scores.get)
print(f"  🏆 {best_model} (ROUGE-L: {scores[best_model]:.4f})")

print("\n✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*100)